In [ ]:
# --- [ CELL 1: SETUP ] ---
#          [TASKS]
# 1. Installs the necessary Python Libraries
# 2. Installs a Chromium browser for Playwright
#     (other browser types did not work)
#####################################

!apt-get update
!apt-get install -y chromium-browser
!pip install playwright pandas bs4
!playwright install ch



Hit:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 3,917 B in 2s (2,469 B/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (

ERROR:asyncio:Future exception was never retrieved
future: <Future finished exception=TargetClosedError('Target page, context or browser has been closed')>
playwright._impl._errors.TargetClosedError: Target page, context or browser has been closed


In [ ]:
# --- [ CELL 2: SCRAPE FROM BASKETBALL REFERENCE WEBSITE ] ---
#                           [TASKS]
# 1. Visits Basketball-Reference monthly schedule pages for each NBA season
# 2. Extracts: date, teams, scores, month number, season label, league
# 3. Captures "markers" so we can drop postseason later
# 4. Writes a single CSV with all rows across the requested seasons
#####################################

import time, csv
import pandas as pd
from bs4 import BeautifulSoup, Comment
from playwright.async_api import async_playwright
import asyncio

BASE = "https://www.basketball-reference.com/leagues"
MONTHS = ["october","november","december","january","february","march","april","may","june","july","august","september"]

START_SEASON = 1948  # change if testing smaller range
END_SEASON   = 2025  # keep small first, later set 2025

def league_for_season(season):
    return "BAA" if season <= 1949 else "NBA"

def month_url(season, month):
    return f"{BASE}/{league_for_season(season)}_{season}_games-{month}.html"

def parse_table(html: str) -> pd.DataFrame:
    soup = BeautifulSoup(html, "html.parser")
    table = soup.find("table", id="schedule")
    if not table:
        for c in soup.find_all(string=lambda t: isinstance(t, Comment)):
            if "<table" in c.lower():
                soup.append(BeautifulSoup(c, "html.parser"))
        table = soup.find("table", id="schedule")
    if not table or not table.tbody:
        return pd.DataFrame()

    rows = []
    for tr in table.tbody.find_all("tr"):
        if "class" in tr.attrs and "thead" in tr["class"]:
            continue
        th = tr.find("th")
        if not th:
            continue
        date_text = th.get_text(strip=True)
        vis  = (tr.find("td", {"data-stat": "visitor_team_name"}) or {}).get_text(strip=True)
        vpts = (tr.find("td", {"data-stat": "visitor_pts"}) or {}).get_text(strip=True)
        home = (tr.find("td", {"data-stat": "home_team_name"}) or {}).get_text(strip=True)
        hpts = (tr.find("td", {"data-stat": "home_pts"}) or {}).get_text(strip=True)
        if not vpts or not hpts:
            continue
        try:
            rows.append({
                "date": pd.to_datetime(date_text).date().isoformat(),
                "visitor_team": vis,
                "visitor_pts": int(vpts),
                "home_team": home,
                "home_pts": int(hpts),
            })
        except Exception:
            continue
    return pd.DataFrame(rows)

async def main():
    all_frames = []
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(user_agent=(
            "Mozilla/5.0 (X11; Linux x86_64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/126.0.0.0 Safari/537.36"
        ))
        page = await context.new_page()

        try:
            for season in range(START_SEASON, END_SEASON + 1):
                for m in MONTHS:
                    url = month_url(season, m)
                    print("GET", url)
                    await page.goto(url, wait_until="domcontentloaded", timeout=60000)
                    html = await page.content()
                    df = parse_table(html)
                    print(f"  parsed rows: {len(df)}")
                    if not df.empty:
                        df.insert(0, "month", m)
                        df.insert(0, "season", season)
                        all_frames.append(df)
                    await asyncio.sleep(1.0)
                await asyncio.sleep(1.5)
        finally:
            await context.close()
            await browser.close()

    if not all_frames:
        print("No data parsed.")
        return

    out = pd.concat(all_frames, ignore_index=True)
    out.to_csv(f"nba_gcl_games_{START_SEASON}_{END_SEASON}.csv", index=False, quoting=csv.QUOTE_MINIMAL)
    print(f"Wrote nba_gcl_games_{START_SEASON}_{END_SEASON}.csv with", len(out), "rows.")

# Run it
await main()


GET https://www.basketball-reference.com/leagues/BAA_1948_games-october.html
  parsed rows: 0
GET https://www.basketball-reference.com/leagues/BAA_1948_games-november.html
  parsed rows: 32
GET https://www.basketball-reference.com/leagues/BAA_1948_games-december.html
  parsed rows: 41
GET https://www.basketball-reference.com/leagues/BAA_1948_games-january.html
  parsed rows: 48
GET https://www.basketball-reference.com/leagues/BAA_1948_games-february.html
  parsed rows: 43
GET https://www.basketball-reference.com/leagues/BAA_1948_games-march.html
  parsed rows: 38
GET https://www.basketball-reference.com/leagues/BAA_1948_games-april.html
  parsed rows: 13
GET https://www.basketball-reference.com/leagues/BAA_1948_games-may.html
  parsed rows: 0
GET https://www.basketball-reference.com/leagues/BAA_1948_games-june.html
  parsed rows: 0
GET https://www.basketball-reference.com/leagues/BAA_1948_games-july.html
  parsed rows: 0
GET https://www.basketball-reference.com/leagues/BAA_1948_games-a

In [ ]:
# --- [ CELL 3: UPLOAD NECESSARY FILES ] ---
#                 [TASKS]
# 1. Allows uploading files
#####################################

from google.colab import files
files.download("nba_gcl_games_1948_2025.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:


# --- [ CELL 4: CLEAN AND STANDARDIZE ] ---
#                           [TASKS]
# 1. Loads scraped CSV or uploaded pre-scraped CSV in Cell 2
# 2. Loads team abbreviation key and maps long names to abbreviations
# 3. Infers correct season from actual game date
# 4. EXCLUDES PLAYOFFS (drop any rows with "Playoffs" in remarks, drop May/June)
# 5. Renames columns to match data dictionary
# 6. Outputs a cleaned CSV
#####################################

# --- Setup ---
import pandas as pd
from google.colab import files
import io

# --- Upload your files (CSV) manually ---
print("📂 Please upload nba_gcl_games_1948_2025.csv (scraper output)")
uploaded_games = files.upload()

print("📂 Please upload NBA_Abbreiviations_Key.csv")
uploaded_abbr = files.upload()

# Read into pandas
games_filename = list(uploaded_games.keys())[0]
abbr_filename  = list(uploaded_abbr.keys())[0]

df = pd.read_csv(io.BytesIO(uploaded_games[games_filename]))
key = pd.read_csv(io.BytesIO(uploaded_abbr[abbr_filename]))

# --- Config ---
INCLUDE_JUNE_AS_PREVIOUS = True

# --- Build mapping (team name/title → abbreviation) ---
key_name_col_candidates = [c for c in key.columns if c.strip().lower() in ("team title", "team name")]
if not key_name_col_candidates:
    raise ValueError(f"Expected a team name column in abbreviations file, found: {list(key.columns)}")
team_name_col = key_name_col_candidates[0]

abbr_col_candidates = [c for c in key.columns if c.strip().lower() == "abbreviation"]
if not abbr_col_candidates:
    raise ValueError(f"Expected an 'Abbreviation' column in abbreviations file, found: {list(key.columns)}")
abbr_col = abbr_col_candidates[0]

# Fix typo
key[team_name_col] = key[team_name_col].astype(str)
key[team_name_col] = key[team_name_col].str.replace(
    r"^\s*SYRACRUSE NATIONALS\s*$", "Syracuse Nationals", case=False, regex=True
)

mapping = dict(zip(
    key[team_name_col].str.strip().str.upper(),
    key[abbr_col].astype(str).str.strip().str.upper()
))

def to_abbr(name: str):
    if pd.isna(name):
        return name
    return mapping.get(str(name).strip().upper(), name)

df["visitor_team"] = df["visitor_team"].apply(to_abbr)
df["home_team"]    = df["home_team"].apply(to_abbr)

# --- Fix season year ---
df["date"] = pd.to_datetime(df["date"], errors="coerce")

def compute_season(ts):
    if pd.isna(ts):
        return None
    y, m = ts.year, ts.month
    if m in (10, 11, 12):
        return y
    if m in (1, 2, 3, 4, 5):
        return y - 1
    if m == 6:
        return y - 1 if INCLUDE_JUNE_AS_PREVIOUS else y
    return y - 1

df["season"] = df["date"].apply(compute_season)

# ---  Drop any rows where season < 1949 ---
df = df[df["season"].fillna(0).astype(int) >= 1949]

# --- Add 'result' and 'home_away' ---
def row_result(row):
    v, h = row["visitor_pts"], row["home_pts"]
    if pd.isna(v) or pd.isna(h):
        return ""
    return "W" if v > h else ("L" if v < h else "")

df["result"] = df.apply(row_result, axis=1)
df["home_away"] = "Home"

# --- Drop & rename columns ---
if "month" in df.columns:
    df = df.drop(columns=["month"])

rename_map = {
    "visitor_team": "team2",
    "visitor_pts": "score2",
    "home_team": "team1",
    "home_pts": "score1"
}
df = df.rename(columns=rename_map)

# Reorder columns correctly
final_cols = ["season", "date", "team1", "team2", "result", "score1", "score2", "home_away"]
df = df[final_cols]

# --- Save & download ---
output_filename = "cleaned_nba_games_1949_2025.csv"
df.to_csv(output_filename, index=False)

print(f"\n✅ Cleaned file saved as {output_filename}")
files.download(output_filename)


📂 Please upload nba_gcl_games_1949_2025.csv (scraper output)


Saving nba_gcl_games_1948_2025.csv to nba_gcl_games_1948_2025 (2).csv
📂 Please upload NBA_Abbreiviations_Key.csv


Saving NBA_Abbreiviations_Key.csv to NBA_Abbreiviations_Key (2).csv

✅ Cleaned file saved as cleaned_nba_games_1949_2025.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>